# 1. Initialization

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Products Silver Pipeline Started")

# 2. Read Bronze subscription_plans

In [0]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/subscription_plans"

df_bronze = (
    spark.read.format("delta").load(bronze_path)
)

df_bronze.show(5)
df_bronze.printSchema()

# 3. Cast and Standardize

In [0]:
df1 = (
    df_bronze.select(
        F.col("plan_id").cast("int"),
        F.col("product_id").cast("int"),
        F.col("tier"),
        F.col("billing_cycle"),
        F.col("price").cast("decimal(18,2)"),
        F.upper(F.col("currency")).alias("currency"),
        F.col("created_at").cast("timestamp"),
        F.col("ingest_time").cast("timestamp"),
        F.col("source_table"),
        F.col("batch_id")
    )
    .withColumn("tier", F.initcap(F.lower(F.col("tier"))))
    .withColumn("billing_cycle", F.lower(F.col("billing_cycle")))
)

# 4. Validation

In [0]:
valid_tiers = ["Free Trial", "Basic", "Standard", "Premium", "Enterprise"]
valid_cycles = ["monthly", "annual"]
valid_currencies = ["USD", "EUR", "GBP", "VND"]

df2 = (
    df1.withColumn(
        "validation_error",
        F.concat_ws(
            ";",
            F.when(F.col("plan_id").isNull(), "plan_id_null"),
            F.when(F.col("product_id").isNull(), "product_id_null"),
            F.when(~F.col("tier").isin(valid_tiers), "invalid_tier"),
            F.when(~F.col("billing_cycle").isin(valid_cycles), "invalid_cycle"),
            F.when(~F.col("currency").isin(valid_currencies), "invalid_currency"),
            F.when(F.col("price") < 0, "invalid_price"),
            F.when(F.col("created_at").isNull(), "missing_created_at")
        )
    )
)

df_valid = df2.filter(F.col("validation_error") == "")
df_quarantine = df2.filter(F.col("validation_error") != "")

# 5. Deduplication

In [0]:
w = Window.partitionBy("plan_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())

df3 = (
    df_valid
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_dup = (
    df_valid
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate_plan"))
)

# 6. Product Integrity Check

In [0]:
product_path = "/Volumes/datalake_catalog/datalake_schema/silver/products"

df_products = (
    spark.read.format("delta").load(product_path)
    .select("product_id").dropDuplicates()
)

df4 = df3.join(df_products, "product_id", "inner")

df_invalid_product = (
    df3.join(df_products, "product_id", "left_anti")
    .withColumn("validation_error", F.lit("invalid_product_id"))
)

# 7. Combine Quarantine

In [0]:
df_quarantine_all = df_quarantine.unionByName(df_dup, True).unionByName(df_invalid_product, True)

# 8. Final Silver Dataset

In [0]:
df_silver = df4

# 9. Write to Silver Delta Lake (Upsert)

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/plans"

w2 = Window.partitionBy("plan_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())

df_upsert = (
    df_silver
    .withColumn("rn", F.row_number().over(w2))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("t").merge(
        df_upsert.alias("s"),
        "t.plan_id = s.plan_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

# 10. Validation Read

In [0]:
df_check = spark.read.format("delta").load(silver_path)
df_check.show(5)
df_check.count()